# **Projekt DIGITS**
## 1. Cel: rozpoznanie cyfr 0-9 z nagrań

---
*Autorzy*:
- *Wojciech Salamak*
- *Amelia Sowa*
- *Szymon Stach*
- *Kamil Kotarba*



## 2. Plan i architektura systemu

### 1. Architektura i przepływ danych

````mermaid
flowchart LR
    n1["Pliki WAV"] --> n2["Odczyt audio i metadanych z nazwy pliku"]
    n2 --> n3["Przygotowanie sygnału"]
    n3 --> n4["Ekstrakcja cech MFCC"]
    n4 --> n5["Trening"] & n6["Klasyfikacja"]

    n1@{ shape: rect}
    n2@{ shape: rounded}
    n3@{ shape: rounded}
    n4@{ shape: rounded}
    n5@{ shape: lin-proc}
    n6@{ shape: lin-proc}
    style n1 fill:transparent,stroke:#FF6D00
    style n2 fill:transparent
    style n3 fill:transparent
    style n4 fill:transparent
    style n5 fill:transparent,stroke:#00C853
    style n6 fill:transparent,stroke:#2962FF
````

Przyjęto architekturę generatywną. Sygnał audio jest opisywany za pomocą współczynników MFCC. Każda cyfra jest reprezentowana przez osobny model GMM. Metoda ta pozwoli na modelowanie rozkładu cech bez wymagania jednakowej długości nagrań.

System jest podzielony na dwie ścieżki: 
- Trening modeli na nagraniach z etykietami w nazwach plików,
- Klasyfikacja nagrania o nieznanej cyfrze

Obie ścieżki wykorzystują tą samą procedurę przygotowania sygnału i ekstrakcji cech.

````mermaid

flowchart LR
    n6["Powiązanie cech z etykietą"] --> n7["Połączenie ramek z tych samych cyfr"]
    n7 --> n8["Trening osobnego GMM cyfr 0-9"]
    n8 --> n9["Słownik modeli GMM"]
    n5["Trening"] --> n11["Cechy MFCC"]
    n11 --> n6
    n10["Klasyfikacja"] --> n13["Cechy MFCC"]
    n16["Wybór najwyższego wyniku"] --> n17["Rozpoznana cyfra 0-9"]
    n18@{ label: "<span style=\"background-color:\">Obliczanie log-podobieństwa przez każdy model</span><br>" } --> n16
    n13 --> n18
    n15["Słownik modeli GMM"] --> n18

    n6@{ shape: rounded}
    n7@{ shape: rounded}
    n8@{ shape: rounded}
    n9@{ shape: db}
    n5@{ shape: lin-proc}
    n11@{ shape: db}
    n10@{ shape: lin-proc}
    n13@{ shape: db}
    n16@{ shape: rounded}
    n18@{ shape: rounded}
    n15@{ shape: db}
    style n9 stroke:#FFD600
    style n5 fill:transparent,stroke:#00C853
    style n11 stroke:#FF6D00
    style n10 stroke:#2962FF
    style n13 stroke:#FF6D00
    style n17 stroke:#00C853
    style n15 stroke:#FFD600
````

### 2. Struktura danych przechowująca cechy MFCC
Cechy ekstraktowane z nagrań będą przechowywane jako lista słowników. Pozwala to na czytelny i łatwy dostęp do identyfikatora mówcy i wypowiedzianej cyfry.
Każdy jej element odpowiada jednemu plikowi audio.


```python
recordings = [
    {
        "path": "training_data/JC1M1_4_.wav",
        "speaker": "JC",
        "digit": 4,
        "features": mfcc_matrix
    }
]
```
Pole `features` zawiera macierz Numpy o wymiarze: (liczba_wspolczynników_MFCC, liczba_ramek).
Każdy wiersz opisuje jedna ramkę czasową. 
<p>Osobna macierz dla każdego nagrania pozwoli przechowywać wypowiedzi rożnej długości.

### 3. Struktura danych przechowująca modele GMM
System wykorzysta dziesięć modeli GMM, jeden dla każdej cyfry. 
Wytrenowane modele będą przechowywane w słowniku `models`

```python
models = {
    0: gmm_0,
    1: gmm_1,
    2: gmm_2,
    # ...
    9: gmm_9,
}
```
Każdy model przechowuje wagi komponentów, ich wektory średnich i macierze kowariancje rozkładu cech danej cyfry. 
Słownik ten zwiąże modele z etykietami klas i umożliwi poźniejszą ocenę nagrania.


### 4. Wstępny plan prac

Plan w formie diagramu Gantta jest na githubie pod zakładce projekt https://github.com/users/engelberd/projects/3

10.11 - 11.11 | Opis pomysłu i architektury | Wojciech Salamak

12.11 | Parametryzacja audio, trening klasyfikatora GMM | Wojciech Salamak, Szymon Stach

15.11 | Podstawowe testy klasyfikatora | Kamil Kotarba

18.11 - 21.11 | Implementacja kroswalidacji | Kamil Kotarba

21.11 - 24.11 | Podział danych według mówców | Wojciech Salamak, Amelia Sowa


25.11 - 27.11 | Strojenie parametrów modeli | Wojciech Salamak, Szymon Stach

28.11 - 29.11 | Test na danych ewaluacyjnych | Amelia Sowa, Szymon Stach



## 3. Przygotwanie danych

### Biblioteki

In [2]:
import numpy as np
from sklearn import mixture
from pathlib import Path
import gdown
import re
import librosa
import pickle
import random
from zipfile import ZipFile

### Funkcje

In [ ]:
def download_zip(zip_url, extract_dir="data"):
    data_dir = Path(extract_dir)
    data_dir.mkdir(parents=True, exist_ok=True)
    zip_path = data_dir / "digits.zip"

    if gdown.download(zip_url, str(zip_path), quiet=False) is None:
        raise RuntimeError("Nie udało się pobrać ZIP-a.")

    with ZipFile(zip_path) as archive:
        archive.extractall(data_dir)

    zip_path.unlink()
    return data_dir

def filter_recs(recordings:list, **kwargs): ### Zwaraca wszystkie słowniki, które mają podaną w parametrze wartość dla danego klucza
    return [
        d for d in recordings 
        if all(d[k] == v for k, v in kwargs.items())
    ]


def train_gmm(mfcc:np.ndarray, n:int, cov_type:str='diag'):
    gmm = mixture.GaussianMixture(n_components=n, covariance_type=cov_type,
                              tol=0.0000001, reg_covar=1e-02, max_iter=100,
                              n_init=1, init_params='random',
                              weights_init=None, means_init=None,precisions_init=None,
                              random_state=16, warm_start=False,
                              verbose=0, verbose_interval=1)
    return gmm.fit(mfcc.T)


def valid(recs:list[dict], valid_models:dict, num_of_tests:int, verbose:int = 0):
    success_no = 0
    fail_no = 0
    for i in range(num_of_tests):
        
        if verbose >= 1:
            print(f"Iteration: {i}\n")
            
        tested = random.randint(0,len(recs)-1)
        tested_file = recs[tested]
        
        if verbose >= 2:
            print(f"File: {tested_file["path"]}\n")
            
        if verbose >= 1:
            print(f"Digit: {tested_file["digit"]}\n")

        results = {}
        for j in valid_models.keys():
            results[j] = round(valid_models[j].score(tested_file["features"].T),2)
            
        if verbose >= 2:
            print(f"Results: \n{results}\n")

        digit = tested_file["digit"]
        digit_guess = max(results, key=results.get)
        
        if verbose >= 1:
            print(f"Digit guess: {digit_guess}\n")
            
        if digit == digit_guess: 
            success_no += 1
            
            if verbose >= 1:
                print("True\n")
                
        else: 
            fail_no += 1
            
            if verbose >= 1:
                print("False\n")
        
    accuracy = round((success_no/num_of_tests)*100,2)

    if verbose >=0:
        print(f"Test with {num_of_tests} iterations ended with {success_no} succesfull guesses and {fail_no} failed guesses {accuracy}% accuracy")

### Pobranie danych

In [ ]:
path = download_zip("https://drive.google.com/file/d/1ODmED_eC1FF4hfy_pQ_rfkzQTFS6Vb_h/view?usp=share_link", extract_dir='training_data')

Downloading...
From: https://drive.google.com/uc?id=1ODmED_eC1FF4hfy_pQ_rfkzQTFS6Vb_h
To: /workspaces/TM-Grupa-6/Project-DIGITS/training_data/digits.zip
100%|██████████| 3.92M/3.92M [00:00<00:00, 112MB/s]


In [3]:
data_dir = Path("training_data")
wav_paths = sorted(
    path for path in data_dir.rglob("*")
    if path.is_file() and path.suffix.lower() == ".wav"
)

recordings = []

for path in wav_paths: 
    match = re.search(r"_([0-9])_", path.stem)

    recordings.append({
        "path": path,
        "speaker": path.name[:2],
        "digit": int(match.group(1)),
    })

## 4. Parametryzacja

In [ ]:
for rec in recordings:
    y, sr = librosa.load(rec["path"], sr=16000)
    
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    
    rec["features"] = mfcc

with open("sparametryzowane_dane.pkl", "wb") as plik:
  pickle.dump(recordings, plik)

## 5. Model klasyfikacyjny


In [5]:
with open("sparametryzowane_dane.pkl", "rb") as plik:
  recordings = pickle.load(plik)
  
models = {}
for i in range(0,10):
    recs = filter_recs(recordings,digit = i)

    conc_mfcc = np.concatenate([d['features'] for d in recs],1)
    
    models[i] = train_gmm(conc_mfcc,4,'diag')


## 6. Walidacja

In [6]:
for i in range(5):
    valid(recs=recordings, valid_models=models, num_of_tests=10)

Test with 10 iterations ended with 9 succesfull guesses and 1 failed guesses 90.0% accuracy
Test with 10 iterations ended with 10 succesfull guesses and 0 failed guesses 100.0% accuracy
Test with 10 iterations ended with 9 succesfull guesses and 1 failed guesses 90.0% accuracy
Test with 10 iterations ended with 9 succesfull guesses and 1 failed guesses 90.0% accuracy
Test with 10 iterations ended with 10 succesfull guesses and 0 failed guesses 100.0% accuracy


## 7. Ewaluacja

## 8. Wnioski